# Advanced ML Pipeline - Cross-Validation & Optimization

We extend a standard pipeline with:

1. Cross-validation
2. Hyperparameter optimization
3. Pipeline abstraction (Scikit-learn)
4. Robust evaluation

Goal:

Build a reliable and reproducible ML workflow.


In [7]:
# DATA LOADING

import numpy as np
import pandas as pd

from sklearn.datasets import load_iris

data = load_iris()

X = data.data
y = data.target

## Why Pipelines?

Without pipeline:

- manual steps
- risk of data leakage
- inconsistent transformations

Pipeline ensures:

    same transformations applied everywhere

### Key Insight

Pipeline = structured data flow

In [8]:
# PIPELINE DEFINITION

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    
    # Step 1: Scaling
    # WHAT: normalize feature magnitudes
    # WHY: ensures stable optimization + distance calculations
    ("scaler", StandardScaler()),
    
    # Step 2: PCA
    # WHAT: reduce dimensionality
    # WHY: remove noise + compress information
    ("pca", PCA()),
    
    # Step 3: Classifier
    # WHAT: final prediction step
    # WHY: model relationship between features and target
    ("clf", LogisticRegression())
])

## Cross-Validation - General Concept and 5-Fold Example

### 1. The Core Problem

When we train a model, we want to know:

    "How well will this model perform on new data?"

### Problem with a Single Train/Test Split

A normal split gives:

    one performance value

This is unreliable because:

- data split is random  
- results depend on which samples are in train/test  
- evaluation can vary significantly  

### 2. Solution: Cross-Validation

Instead of one experiment:

--> perform multiple experiments  

### General Idea

1. Split dataset into K parts (folds)  
2. Train model K times  
3. Each time:
   - use K−1 folds for training  
   - use 1 fold for testing  
4. Average all results  

### Result

Instead of one score:

--> multiple scores --> more reliable estimate  

### How? Example 5-Fold-Cross-Validation

We split the set into 5 equal parts.

Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5

We perform 5 training cycles:

#### Run 1:
Train: Fold 2–5  
Test:  Fold 1  

#### Run 2:
Train: Fold 1,3,4,5  
Test:  Fold 2  

#### Run 3:
Train: Fold 1,2,4,5  
Test:  Fold 3  

#### Run 4:
Train: Fold 1,2,3,5  
Test:  Fold 4  

#### Run 5:
Train: Fold 1–4  
Test:  Fold 5  

### Why This Works

Each data point is:

- used multiple times for training  
- used exactly once for testing  

### Key Effect

- reduces randomness  
- avoids lucky/unlucky splits  
- gives more stable and realistic performance  

## 3. Scientific Interpretation

Instead of:

    one measurement

we perform:

    repeated experiments

--> similar to replicating experiments in science  

## 4. Trade-Offs

More folds (e.g. 10):

- more accurate estimate
- more computation

Fewer folds:

- faster 
- less stable  

## 5. When to Use

Always use cross-validation when:

- dataset is limited  
- results must be reliable  
- tuning model parameters  

## 6. Important Insight

Cross-validation does NOT improve the model directly.

It improves:

    how reliably we evaluate the model

## 7. Common Mistakes

- using CV incorrectly with data leakage  
- scaling data outside pipeline  
- interpreting CV score as “perfect truth”  

## 8. Final Takeaway

Cross-validation =

    multiple train/test splits
    --> average performance
    --> reliable estimate of real-world behavior

In [9]:
# CROSS-VALIDATION

from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=5,            # 5-fold cross-validation
    scoring="accuracy"
)

print("Scores:", scores)
print("Mean accuracy:", np.mean(scores))

Scores: [0.96666667 1.         0.93333333 0.9        1.        ]
Mean accuracy: 0.9600000000000002


## Hyperparameter Optimization

Model parameters (weights):
    learned during training

Hyperparameters:
    chosen before training

### What GridSearch does

- tries all combinations
- evaluates each using cross-validation
- selects best performing setup

### Key Insight

We optimize:

    model structure, not just weights

In [10]:
# GRID SEARCH (HYPERPARAMETER OPTIMIZATION)

from sklearn.model_selection import GridSearchCV

param_grid = {
    
    # PCA parameters
    # WHY: choose best dimensionality
    "pca__n_components": [2, 3, 4],
    
    # Regularization strength
    # WHY: prevent overfitting
    "clf__C": [0.1, 1.0, 10.0],
    
    # Optimization algorithm
    # WHY: affects convergence behavior
    "clf__solver": ["lbfgs"]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    
    cv=5,               # cross-validation
    scoring="accuracy",
    n_jobs=-1           # parallel execution
)

grid.fit(X, y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'clf__C': [0.1, 1.0, ...], 'clf__solver': ['lbfgs'], 'pca__n_components': [2, 3, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Contr

In [11]:
print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Best Parameters: {'clf__C': 10.0, 'clf__solver': 'lbfgs', 'pca__n_components': 3}
Best CV Score: 0.9733333333333334


## Result Interpretation

best_params_:

--> optimal configuration

best_score_:

--> average performance across folds

### Important

Cross-validation score is more reliable than:

    single train-test split


In [12]:
#EVALUATION
from sklearn.metrics import classification_report

best_model = grid.best_estimator_

preds = best_model.predict(X)

print(classification_report(y, preds))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        50
           1       0.98      0.94      0.96        50
           2       0.94      0.98      0.96        50

    accuracy                           0.97       150
   macro avg       0.97      0.97      0.97       150
weighted avg       0.97      0.97      0.97       150



## Full Optimization Process

Pipeline consists of multiple optimizations:

Scaling:
    normalizes data (no optimization)

PCA:
    maximizes variance


Logistic Regression:
    minimizes classification loss


GridSearch:
    selects best configuration


### Key Insight

Optimization happens at multiple levels:

1. Data transformation
2. Feature extraction
3. Model learning
4. Hyperparameter tuning

## Advanced Exercises

### Cross-validation

1. Change cv=10 --> what happens?

### PCA

2. Restrict to n_components=1  
3. Compare performance  

### Regularization

4. Increase C --> what happens?  
5. Decrease C --> what happens?  

### Pipeline

6. Remove PCA --> compare results  

### Optimization

7. Add more parameters to GridSearch  
8. Does better tuning always improve performance?  

### Critical Thinking

9. Can cross-validation overestimate performance?  

### Bonus

10. Use F1-score instead of accuracy  